In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"])

In [2]:
import sys, os
from dotenv import load_dotenv

# Load .env from project root
load_dotenv(os.path.abspath('../.env'))

sys.path.insert(0, os.path.abspath('..'))

# Clear any cached src modules so edits to .py files take effect immediately
for key in list(sys.modules.keys()):
    if key.startswith("src"):
        del sys.modules[key]

for var in ("OPENAI_API_KEY", "GOOGLE_MAPS_API_KEY", "MAPBOX_API_KEY"):
    status = "✓ set" if os.environ.get(var) else "✗ not set"
    print(f"{var}: {status}")

OPENAI_API_KEY: ✓ set
GOOGLE_MAPS_API_KEY: ✓ set
MAPBOX_API_KEY: ✓ set


In [ ]:
from src.ground_truth import build_ground_truth

gt_df = build_ground_truth(
    provider="openai",
    model="gpt-4o-mini",   # 10x cheaper than gpt-4o; switch to "gpt-4o" for best quality
    sample_n=750,
    tile_workers=8,        # parallel tile fetching
    # max_places=5,       # uncomment to test on a small batch first
)

Sampled 750 places from 2050 strata (target: 750)
Fetching 750 tiles with 8 workers...
  [TILE 1/750] 08f2a92d8dd049a1037a554a7fba4de1 ✓
  [TILE 2/750] 08f275802b50211203f9d9bc9df97762 ✓
  [TILE 3/750] 08f26199718ee09803fa3523ec4bb6d7 ✓
  [TILE 4/750] 08f2aaa84636d40303c03a120f5005e4 ✓
  [TILE 5/750] 08f28320ec3226580341f3559baa4b57 ✓
  [TILE 6/750] 08f2a924959628cd03f6694fef5aca80 ✓
  [TILE 7/750] 08f29a42b04f362b033906064c2fd0ef ✓
  [TILE 8/750] 08f28843989708a803d12abc8c89a07c ✓
  [TILE 9/750] 08f446c1763a845b032f80ded1ab6217 ✓
  [TILE 10/750] 08f2ab38e6c9560403b1b8b59c797291 ✓
  [TILE 11/750] 08f44f6a95c9b02103105fd40c76bfe4 ✓
  [TILE 12/750] 08f2aa2dae58bbb003c7483acb4674bd ✓
  [TILE 13/750] 08f2a100dedae419036be0a1f14b7d61 ✓
  [TILE 14/750] 08f44a9ba233292603ba51909069e795 ✓
  [TILE 15/750] 08f4899793ce20a50368903a9994970e ✓
  [TILE 16/750] 08f2a33076945d9903d177270b372474 ✓
  [TILE 17/750] 08f2aa8c744b0c80039656f47402c67c ✓
  [TILE 18/750] 08f28843556560f303f2d601df458d20 ✓
  [T

In [ ]:
from src.cost_tracker import print_summary

print_summary()

In [ ]:
gt_df[["name", "category_primary", "region",
       "gt_confidence", "gt_reasoning",
       "offset_haversine_m", "offset_euclidean_m", "offset_manhattan_m"]].head(10)

In [ ]:
# Helps verify that the tiles were downloaded correctly and aren't corrupted (sometimes happens with Mapbox API). No need to run everytime
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

tiles_dir = Path("../data/processed/tiles")
tile_files = sorted(tiles_dir.glob("*.png"))[:5]

good = []
for p in tile_files:
    try:
        img = Image.open(p)
        img.verify()
        good.append(p)
    except Exception:
        print(f"Corrupt tile (will be re-fetched): {p.name}")
        p.unlink()

if not good:
    print("No valid tiles yet — re-run the ground truth cell")
else:
    fig, axes = plt.subplots(1, len(good), figsize=(4 * len(good), 4))
    if len(good) == 1:
        axes = [axes]
    for ax, p in zip(axes, good):
        ax.imshow(Image.open(p))
        ax.set_title(p.stem[:16], fontsize=7)
        ax.axis("off")
    plt.suptitle("Satellite tiles — red dot = current pin", fontsize=10)
    plt.tight_layout()
    plt.show()